# TurboQuant KV Cache Compression — Benchmark Experiment

**Model**: `VAGOsolutions/Llama-3.1-SauerkrautLM-8b-Instruct`  
**Hardware**: Single RTX 3090 (24 GB VRAM)  
**Paper**: TurboQuant (ICLR 2026, arXiv:2504.19874)

---

## What this notebook does

This notebook measures the effect of **TurboQuant KV cache compression** on a Llama 3.1 8B model. TurboQuant compresses the key-value cache that accumulates during inference from BF16 (~16 bits) down to ~3 bits for keys and ~2 bits for values, achieving roughly 2.6× compression per attention layer.

We compare two inference modes — **Baseline** (standard BF16 KV cache) and **TurboQuant** (compressed KV cache) — across three context lengths: 2K, 8K, and 32K tokens.

**Metrics we measure:**
- Peak VRAM usage (how much GPU memory does each approach use?)
- Perplexity on WikiText-2 (does compression hurt quality?)
- Time-to-first-token / decode throughput (what's the latency cost?)
- Maximum context before out-of-memory (how much more context can we fit?)

**Prerequisites:** Run `bash setup.sh` before launching this notebook.

## Section 0: Setup & Environment Check

Before running any benchmarks, we verify that all dependencies are installed and the GPU is what we expect. Every cell in this section should print green results. If anything fails, run `bash setup.sh` again.

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Check Python package versions
import importlib

packages = {
    "torch": "torch",
    "vllm": "vllm",
    "turboquant": "turboquant",
    "transformers": "transformers",
    "datasets": "datasets",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
}

all_ok = True
for display_name, import_name in packages.items():
    try:
        mod = importlib.import_module(import_name)
        version = getattr(mod, "__version__", "unknown")
        print(f"  OK  {display_name:<15} {version}")
    except ImportError:
        print(f"  MISSING  {display_name}  — run setup.sh")
        all_ok = False

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram_gb = gpu.total_memory / 1024**3
    print(f"\n  GPU:  {gpu.name}  ({vram_gb:.1f} GB VRAM)")
    if "3090" not in gpu.name:
        print("  WARNING: Expected RTX 3090. Results may differ from reference.")
else:
    print("  ERROR: No CUDA GPU found")
    all_ok = False

print("\nAll OK" if all_ok else "\nSome packages missing — run setup.sh")

In [ ]:
# Check that bench/ modules are importable
import sys, os
# Add repo root to path (one level up from notebook/)
repo_root = os.path.dirname(os.getcwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from bench import engine, perplexity, memory, latency, oom_search, subprocess_runner
print("bench/ modules imported successfully")

In [ ]:
# Experiment configuration — all tunable parameters in one place
CONFIG = {
    "model_id": "VAGOsolutions/Llama-3.1-SauerkrautLM-8b-Instruct",
    "dtype": "bfloat16",
    "gpu_memory_utilization": 0.95,   # MUST be identical for baseline and TQ
    "tensor_parallel_size": 1,         # Single GPU

    "context_lengths": [2048, 8192, 32768],  # Tokens
    "num_generate_tokens": 256,               # Tokens to generate per run
    "num_repetitions": 3,                     # Take median of 3 for stability

    # TurboQuant settings
    "tq_key_bits": 3,
    "tq_value_bits": 2,

    # Dataset
    "perplexity_dataset": "wikitext",
    "perplexity_subset": "wikitext-2-raw-v1",

    # Output
    "results_dir": "../results",
}

print("Experiment configuration:")
print(f"  Model:           {CONFIG['model_id']}")
print(f"  Context lengths: {CONFIG['context_lengths']}")
print(f"  Repetitions:     {CONFIG['num_repetitions']}")
print(f"  TQ bits:         {CONFIG['tq_key_bits']}b keys / {CONFIG['tq_value_bits']}b values")
print(f"  GPU util:        {CONFIG['gpu_memory_utilization']}")
print()
print(f"We will test {len(CONFIG['context_lengths'])} context lengths × "
      f"{CONFIG['num_repetitions']} repetitions × 2 modes = "
      f"{len(CONFIG['context_lengths']) * CONFIG['num_repetitions'] * 2} total runs")

## Section 1: Understanding the KV Cache Problem

### What is the KV cache?

During transformer inference, each attention layer computes **key** and **value** tensors for every input token. These are cached so that subsequent tokens don't need to recompute them — this is the KV cache. Without it, generating token N would require O(N²) compute.

The problem: **the KV cache grows linearly with context length**. At BF16 (2 bytes per element), for a model with 32 layers, 8 KV heads, and head dimension 128:

$$\text{KV size} = 32 \text{ layers} \times 2 \text{ (K+V)} \times 8 \text{ KV heads} \times 128 \text{ head\_dim} \times N \text{ tokens} \times 2 \text{ bytes}$$

On an RTX 3090 with 24 GB VRAM, the model weights take ~16 GB, leaving only ~8 GB for the KV cache. At 32K tokens, the KV cache alone would need 4 GB — a third of the remaining budget.

### What does TurboQuant do?

TurboQuant compresses the KV cache from BF16 (~16 bits) to ~3 bits for keys and ~2 bits for values. It does this using:
1. A random orthogonal rotation to spread information across dimensions
2. Lloyd-Max optimal scalar quantization on the rotated keys
3. Group quantization for values

After prefill, TurboQuant frees the original paged KV tensors allocated by vLLM, keeping only the compressed representation. The result: **2–5× less VRAM for the KV cache**, meaning we can fit longer contexts on the same GPU.

In [ ]:
# Theoretical KV cache sizes for Llama 3.1 8B at each context length
import pandas as pd

LAYERS = 32
KV_HEADS = 8
HEAD_DIM = 128
DTYPE_BYTES = 2  # BF16

# TQ compression: 3-bit keys + 2-bit values → effective ~2.5 bits/element
# Plus overhead for norms, scales, indices (~15% overhead estimated)
TQ_BITS_PER_ELEMENT = 2.5 * 1.15  # effective bits including overhead

rows = []
for ctx in CONFIG["context_lengths"]:
    bf16_bytes = LAYERS * 2 * KV_HEADS * HEAD_DIM * ctx * DTYPE_BYTES
    tq_bytes = LAYERS * 2 * KV_HEADS * HEAD_DIM * ctx * (TQ_BITS_PER_ELEMENT / 8)
    rows.append({
        "Context Length": f"{ctx:,}",
        "KV Cache (BF16)": f"{bf16_bytes / 1024**2:.0f} MB",
        f"KV Cache (TQ ~{TQ_BITS_PER_ELEMENT:.1f}bit)": f"{tq_bytes / 1024**2:.0f} MB",
        "Theoretical Savings": f"{(bf16_bytes - tq_bytes) / 1024**2:.0f} MB",
        "Compression Ratio": f"{bf16_bytes / tq_bytes:.1f}×",
    })

df = pd.DataFrame(rows)
display(df.to_string(index=False))
print()
print("Note: TQ savings shown above are theoretical. We will measure actual savings empirically.")

## Section 2: Load the Dataset

We use **WikiText-2** for perplexity measurement. It's the standard benchmark for language model perplexity — natural English text from Wikipedia articles, widely used and well-understood.

**Why not synthetic text?** Repeated or random text would give artificially low perplexity (the model learns the pattern after a few tokens). Natural text gives a realistic measurement of how well the model understands continuous language.

**How we prepare prompts:** We tokenize the entire WikiText-2 test split, concatenate all tokens, then take the first N tokens for context length N. Decoding back to text gives us a clean, consistent string that the model sees identically across runs.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

print("Loading WikiText-2 test split...")
dataset = load_dataset(
    CONFIG["perplexity_dataset"],
    CONFIG["perplexity_subset"],
    split="test",
)
print(f"  Rows: {len(dataset)}")

print(f"Loading tokenizer for {CONFIG['model_id']}...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_id"])

# Concatenate all non-empty rows
full_text = " ".join(row["text"] for row in dataset if row["text"].strip())
all_token_ids = tokenizer.encode(full_text, add_special_tokens=False)

print(f"  Total tokens available: {len(all_token_ids):,}")
print(f"  Max context we need:    {max(CONFIG['context_lengths']):,}")
assert len(all_token_ids) >= max(CONFIG["context_lengths"]), \
    "Not enough tokens in WikiText-2 for the requested context lengths!"
print("  Sufficient tokens available.")

In [ ]:
def get_prompt(num_tokens: int) -> str:
    """
    Return the WikiText-2 test text decoded from exactly num_tokens tokens.
    
    Decoding back from token IDs (rather than slicing characters) ensures
    the tokenizer sees the exact same tokens when it re-encodes for inference.
    """
    ids = all_token_ids[:num_tokens]
    return tokenizer.decode(ids, skip_special_tokens=True)

# Demonstrate
for ctx in CONFIG["context_lengths"]:
    prompt = get_prompt(ctx)
    actual_tokens = len(tokenizer.encode(prompt, add_special_tokens=False))
    print(f"  ctx={ctx:,}: {actual_tokens:,} tokens, first 80 chars: {prompt[:80]!r}")

## Section 3: Baseline Measurements (BF16, No Compression)

We now run inference at each context length using **standard vLLM** — no TurboQuant, just the model in BF16 with the default paged KV cache.

### Why subprocess isolation?

vLLM allocates its paged KV cache at engine startup, sized for the maximum context length you declare (`max_model_len`). **It does not release this allocation while the engine is alive.** If we reused one engine across context lengths:
- The KV pool would be sized for whichever context we initialized first
- VRAM readings for other context lengths would be wrong
- We'd be comparing apples to oranges

The solution: each measurement runs in a **fresh Python subprocess**. The subprocess creates a new engine (KV pool sized exactly for this context), takes all measurements, writes JSON to stdout, and exits — fully releasing VRAM. The notebook cell reads the JSON.

**Each cell below takes ~10-15 minutes per context length.** Progress is printed after each run.

In [ ]:
%%time
import statistics
from bench.subprocess_runner import run_single_benchmark

print("Starting BASELINE measurements...")
print(f"  {len(CONFIG['context_lengths'])} context lengths × {CONFIG['num_repetitions']} reps")
print()

baseline_results = []  # list of result dicts

for ctx_len in CONFIG["context_lengths"]:
    print(f"--- Context length: {ctx_len:,} tokens ---")
    reps = []
    for rep in range(CONFIG["num_repetitions"]):
        print(f"  Rep {rep + 1}/{CONFIG['num_repetitions']}...", end=" ", flush=True)
        r = run_single_benchmark(CONFIG, context_length=ctx_len, use_tq=False)
        if "error" in r:
            print(f"FAILED: {r['error'][:120]}")
        else:
            print(f"ppl={r['perplexity']:.2f}, vram={r['peak_vram_mb']} MB, "
                  f"ttft={r['ttft_s']:.2f}s, tps={r['decode_tps']:.1f}")
            reps.append(r)

    if reps:
        # Median across repetitions
        median = {
            "context_length": ctx_len,
            "use_tq": False,
            "perplexity": statistics.median(r["perplexity"] for r in reps),
            "peak_vram_mb": statistics.median(r["peak_vram_mb"] for r in reps),
            "ttft_s": statistics.median(r["ttft_s"] for r in reps),
            "decode_tps": statistics.median(r["decode_tps"] for r in reps),
            "raw_reps": reps,
        }
        baseline_results.append(median)
        print(f"  Median: ppl={median['perplexity']:.2f}, vram={median['peak_vram_mb']} MB")
    print()

print("Baseline measurements complete.")

In [ ]:
# Display baseline results as a DataFrame
import pandas as pd

df_baseline = pd.DataFrame([
    {
        "Context Length": f"{r['context_length']:,}",
        "Perplexity": f"{r['perplexity']:.3f}",
        "Peak VRAM (MB)": f"{r['peak_vram_mb']:,}",
        "TTFT (s)": f"{r['ttft_s']:.3f}",
        "Decode (tok/s)": f"{r['decode_tps']:.1f}",
    }
    for r in baseline_results
])

print("BASELINE RESULTS (medians)")
print(df_baseline.to_string(index=False))

In [ ]:
# Quick look at baseline VRAM usage
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax = plt.subplots(figsize=(7, 4))

ctx_labels = [f"{r['context_length'] // 1024}K" for r in baseline_results]
vram_vals = [r["peak_vram_mb"] for r in baseline_results]

bars = ax.bar(ctx_labels, vram_vals, color="#3266ad", alpha=0.85, width=0.5)
ax.axhline(24576, color="red", linestyle="--", linewidth=1.5, label="RTX 3090 limit (24 GB)")

for bar, val in zip(bars, vram_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 200, f"{val:,} MB",
            ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_xlabel("Context Length", fontsize=12)
ax.set_ylabel("Peak VRAM (MB)", fontsize=12)
ax.set_title("Baseline VRAM Usage by Context Length", fontsize=13, fontweight="bold")
ax.legend()
ax.set_ylim(0, 26000)
plt.tight_layout()
plt.show()

## Section 4: TurboQuant Measurements

Now we repeat **exactly the same experiment** with TurboQuant enabled. The only difference is that after prefill, TurboQuant:
1. Captures all KV pairs from vLLM's paged cache
2. Compresses them (3-bit keys via Lloyd-Max codebook, 2-bit values via group quantization)
3. Stores the compressed representation in its own data structures
4. Frees the original paged KV tensors, reclaiming VRAM
5. During decode, reconstructs approximate attention scores from the compressed store

**Same model, same prompts, same config.** The ONLY variable is TurboQuant.

### Adaptation note

The 0xSero TurboQuant repo was tested on Qwen3.5 models. For Llama 3.1 8B, TurboQuant's hook registration detects standard Flash Attention layers automatically — Llama 3.1 uses GQA (grouped-query attention) which vLLM runs through its Flash Attention backend, so TQ should hook all 32 layers. If it hooks 0 layers, an error dict is returned with the model's attention module names for debugging.

In [ ]:
%%time
print("Starting TURBOQUANT measurements...")
print(f"  Key bits: {CONFIG['tq_key_bits']}, Value bits: {CONFIG['tq_value_bits']}")
print()

tq_results = []

for ctx_len in CONFIG["context_lengths"]:
    print(f"--- Context length: {ctx_len:,} tokens ---")
    reps = []
    for rep in range(CONFIG["num_repetitions"]):
        print(f"  Rep {rep + 1}/{CONFIG['num_repetitions']}...", end=" ", flush=True)
        r = run_single_benchmark(CONFIG, context_length=ctx_len, use_tq=True)
        if "error" in r:
            print(f"FAILED: {r['error'][:120]}")
            if "hooked 0 layers" in str(r.get("error", "")):
                print("  TQ hook registration failed. Check attention module names above.")
                print("  You may need to update turboquant/integration/vllm.py for Llama.")
        else:
            print(f"ppl={r['perplexity']:.2f}, vram={r['peak_vram_mb']} MB, "
                  f"ttft={r['ttft_s']:.2f}s, tps={r['decode_tps']:.1f}, "
                  f"hooks={r['num_hooked_layers']}, freed={r['tq_bytes_freed'] // 1024**2} MB")
            reps.append(r)

    if reps:
        median = {
            "context_length": ctx_len,
            "use_tq": True,
            "perplexity": statistics.median(r["perplexity"] for r in reps),
            "peak_vram_mb": statistics.median(r["peak_vram_mb"] for r in reps),
            "ttft_s": statistics.median(r["ttft_s"] for r in reps),
            "decode_tps": statistics.median(r["decode_tps"] for r in reps),
            "num_hooked_layers": reps[0]["num_hooked_layers"],
            "tq_bytes_freed": statistics.median(r["tq_bytes_freed"] for r in reps),
            "raw_reps": reps,
        }
        tq_results.append(median)
        freed_mb = median["tq_bytes_freed"] / 1024**2
        print(f"  Median: ppl={median['perplexity']:.2f}, vram={median['peak_vram_mb']} MB, "
              f"freed={freed_mb:.0f} MB")
    print()

print("TurboQuant measurements complete.")

In [ ]:
# Display TQ results
df_tq = pd.DataFrame([
    {
        "Context Length": f"{r['context_length']:,}",
        "Perplexity": f"{r['perplexity']:.3f}",
        "Peak VRAM (MB)": f"{r['peak_vram_mb']:,}",
        "TTFT (s)": f"{r['ttft_s']:.3f}",
        "Decode (tok/s)": f"{r['decode_tps']:.1f}",
        "Layers Hooked": r.get("num_hooked_layers", "?"),
        "KV Freed (MB)": f"{r.get('tq_bytes_freed', 0) / 1024**2:.0f}",
    }
    for r in tq_results
])

print("TURBOQUANT RESULTS (medians)")
print(df_tq.to_string(index=False))

## Section 5: OOM Ceiling Test

The headline question: **how much MORE context can we fit on the same GPU with TurboQuant?**

We use binary search to find the maximum context length that doesn't crash with OOM. Each attempt runs in a subprocess so OOM kills the child process, not this notebook kernel. The search converges in ~7 iterations (log₂(131072/4096) ≈ 5 doublings, plus refinement).

This is the most impactful metric because it directly answers: *does TurboQuant let me run longer documents, longer conversations, or larger RAG contexts on my existing hardware?*

In [ ]:
%%time
from bench.oom_search import find_max_context

print("Searching for max context (Baseline)...")
max_ctx_baseline = find_max_context(CONFIG, use_tq=False)

print()
print("Searching for max context (TurboQuant)...")
max_ctx_tq = find_max_context(CONFIG, use_tq=True)

multiplier = max_ctx_tq / max_ctx_baseline if max_ctx_baseline > 0 else float("nan")

print()
print("═" * 51)
print("  MAX CONTEXT BEFORE OOM")
print(f"  Baseline:    {max_ctx_baseline:>10,} tokens")
print(f"  TurboQuant:  {max_ctx_tq:>10,} tokens  ({multiplier:.2f}× more)")
print("═" * 51)

## Section 6: Analysis & Visualization

Let's bring all the numbers together and see what TurboQuant actually delivers.

In [ ]:
# Build lookup dicts for easy access
bl = {r["context_length"]: r for r in baseline_results}
tq = {r["context_length"]: r for r in tq_results}
ctx_lengths = [c for c in CONFIG["context_lengths"] if c in bl and c in tq]

print(f"Context lengths with both baseline and TQ data: {ctx_lengths}")

In [ ]:
# ── Plot 1: VRAM vs Context Length ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

BL_COLOR = "#3266ad"
TQ_COLOR = "#1D9E75"

bl_vram = [bl[c]["peak_vram_mb"] for c in ctx_lengths]
tq_vram = [tq[c]["peak_vram_mb"] for c in ctx_lengths]
x_labels = [f"{c // 1024}K" for c in ctx_lengths]
x = range(len(ctx_lengths))

ax.plot(x, bl_vram, color=BL_COLOR, marker="o", linewidth=2, markersize=8, label="Baseline (BF16)")
ax.plot(x, tq_vram, color=TQ_COLOR, marker="s", linewidth=2, markersize=8, label="TurboQuant")
ax.fill_between(x, tq_vram, bl_vram, alpha=0.15, color=TQ_COLOR, label="VRAM savings")

# RTX 3090 VRAM limit
ax.axhline(24576, color="red", linestyle="--", linewidth=1.5, label="RTX 3090 limit (24 GB)")

# Annotate savings at 32K
if ctx_lengths and ctx_lengths[-1] == 32768:
    saved = bl_vram[-1] - tq_vram[-1]
    pct = 100 * saved / bl_vram[-1] if bl_vram[-1] else 0
    ax.annotate(
        f"{saved:,} MB saved\n({pct:.0f}%)",
        xy=(len(ctx_lengths) - 1, (bl_vram[-1] + tq_vram[-1]) / 2),
        xytext=(len(ctx_lengths) - 1.6, (bl_vram[-1] + tq_vram[-1]) / 2),
        arrowprops=dict(arrowstyle="->", color="gray"),
        fontsize=10, color="gray",
    )

ax.set_xticks(list(x))
ax.set_xticklabels(x_labels, fontsize=12)
ax.set_xlabel("Context Length", fontsize=12)
ax.set_ylabel("Peak VRAM (MB)", fontsize=12)
ax.set_title("Peak VRAM Usage: Baseline vs TurboQuant", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig(f"{CONFIG['results_dir']}/plot_vram_vs_context.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Plot 2: Perplexity Comparison ────────────────────────────────────────────
import numpy as np

fig, ax = plt.subplots(figsize=(8, 5))

n = len(ctx_lengths)
bar_width = 0.35
x = np.arange(n)

bl_ppl = [bl[c]["perplexity"] for c in ctx_lengths]
tq_ppl = [tq[c]["perplexity"] for c in ctx_lengths]

# Error bars: min and max from raw reps
def ppl_err(results_dict, ctx):
    raw = results_dict[ctx].get("raw_reps", [])
    if not raw:
        return 0, 0
    vals = [r["perplexity"] for r in raw if "error" not in r]
    med = statistics.median(vals)
    return med - min(vals), max(vals) - med

bl_errs = [ppl_err(bl, c) for c in ctx_lengths]
tq_errs = [ppl_err(tq, c) for c in ctx_lengths]

ax.bar(x - bar_width / 2, bl_ppl, bar_width, label="Baseline", color=BL_COLOR, alpha=0.85,
       yerr=[[e[0] for e in bl_errs], [e[1] for e in bl_errs]], capsize=4, error_kw=dict(elinewidth=1.5))
ax.bar(x + bar_width / 2, tq_ppl, bar_width, label="TurboQuant", color=TQ_COLOR, alpha=0.85,
       yerr=[[e[0] for e in tq_errs], [e[1] for e in tq_errs]], capsize=4, error_kw=dict(elinewidth=1.5))

# Annotate delta
for i, (b, t) in enumerate(zip(bl_ppl, tq_ppl)):
    delta = t - b
    sign = "+" if delta >= 0 else ""
    ax.text(x[i] + bar_width / 2, t + 0.05, f"{sign}{delta:.2f}",
            ha="center", va="bottom", fontsize=9, color="#555")

ax.set_xticks(x)
ax.set_xticklabels([f"{c // 1024}K" for c in ctx_lengths], fontsize=12)
ax.set_xlabel("Context Length", fontsize=12)
ax.set_ylabel("Perplexity (lower is better)", fontsize=12)
ax.set_title("Perplexity: Baseline vs TurboQuant", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f"{CONFIG['results_dir']}/plot_perplexity.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Plot 3: Latency Breakdown ─────────────────────────────────────────────────
fig, (ax_ttft, ax_tps) = plt.subplots(1, 2, figsize=(12, 5))

bl_ttft = [bl[c]["ttft_s"] for c in ctx_lengths]
tq_ttft = [tq[c]["ttft_s"] for c in ctx_lengths]
bl_tps = [bl[c]["decode_tps"] for c in ctx_lengths]
tq_tps = [tq[c]["decode_tps"] for c in ctx_lengths]

# TTFT
ax_ttft.bar(x - bar_width / 2, bl_ttft, bar_width, label="Baseline", color=BL_COLOR, alpha=0.85)
ax_ttft.bar(x + bar_width / 2, tq_ttft, bar_width, label="TurboQuant", color=TQ_COLOR, alpha=0.85)
for i, (b, t) in enumerate(zip(bl_ttft, tq_ttft)):
    overhead_pct = 100 * (t - b) / b if b > 0 else 0
    sign = "+" if overhead_pct >= 0 else ""
    ax_ttft.text(x[i] + bar_width / 2, t + 0.002, f"{sign}{overhead_pct:.0f}%",
                 ha="center", va="bottom", fontsize=9)
ax_ttft.set_xticks(x)
ax_ttft.set_xticklabels([f"{c // 1024}K" for c in ctx_lengths], fontsize=11)
ax_ttft.set_xlabel("Context Length", fontsize=11)
ax_ttft.set_ylabel("Time to First Token (s)", fontsize=11)
ax_ttft.set_title("TTFT: Baseline vs TurboQuant", fontsize=12, fontweight="bold")
ax_ttft.legend(fontsize=10)

# Decode tok/s
ax_tps.bar(x - bar_width / 2, bl_tps, bar_width, label="Baseline", color=BL_COLOR, alpha=0.85)
ax_tps.bar(x + bar_width / 2, tq_tps, bar_width, label="TurboQuant", color=TQ_COLOR, alpha=0.85)
for i, (b, t) in enumerate(zip(bl_tps, tq_tps)):
    delta_pct = 100 * (t - b) / b if b > 0 else 0
    sign = "+" if delta_pct >= 0 else ""
    ax_tps.text(x[i] + bar_width / 2, t + 0.5, f"{sign}{delta_pct:.0f}%",
                ha="center", va="bottom", fontsize=9)
ax_tps.set_xticks(x)
ax_tps.set_xticklabels([f"{c // 1024}K" for c in ctx_lengths], fontsize=11)
ax_tps.set_xlabel("Context Length", fontsize=11)
ax_tps.set_ylabel("Decode Throughput (tok/s)", fontsize=11)
ax_tps.set_title("Decode Throughput: Baseline vs TurboQuant", fontsize=12, fontweight="bold")
ax_tps.legend(fontsize=10)

plt.tight_layout()
plt.savefig(f"{CONFIG['results_dir']}/plot_latency.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Plot 4: Summary Dashboard (2×2) ──────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle("TurboQuant vs Baseline — Summary Dashboard", fontsize=15, fontweight="bold", y=1.01)

x_labels_short = [f"{c // 1024}K" for c in ctx_lengths]

# Top-left: VRAM savings
ax = axes[0, 0]
ax.plot(x, bl_vram, color=BL_COLOR, marker="o", linewidth=2, label="Baseline")
ax.plot(x, tq_vram, color=TQ_COLOR, marker="s", linewidth=2, label="TurboQuant")
ax.fill_between(x, tq_vram, bl_vram, alpha=0.15, color=TQ_COLOR)
ax.axhline(24576, color="red", linestyle="--", linewidth=1, label="24 GB limit")
ax.set_xticks(list(x)); ax.set_xticklabels(x_labels_short)
ax.set_ylabel("VRAM (MB)"); ax.set_title("Peak VRAM"); ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))

# Top-right: Perplexity delta
ax = axes[0, 1]
ppl_deltas = [tq[c]["perplexity"] - bl[c]["perplexity"] for c in ctx_lengths]
colors = ["#e55" if d > 0 else "#1D9E75" for d in ppl_deltas]
ax.bar(x_labels_short, ppl_deltas, color=colors, alpha=0.85)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Perplexity Delta (TQ − Baseline)")
ax.set_title("Perplexity Overhead")
for i, d in enumerate(ppl_deltas):
    sign = "+" if d >= 0 else ""
    ax.text(i, d + 0.005 if d >= 0 else d - 0.01, f"{sign}{d:.3f}",
            ha="center", fontsize=10, fontweight="bold")

# Bottom-left: Throughput comparison
ax = axes[1, 0]
ax.bar(x - bar_width / 2, bl_tps, bar_width, label="Baseline", color=BL_COLOR, alpha=0.85)
ax.bar(x + bar_width / 2, tq_tps, bar_width, label="TurboQuant", color=TQ_COLOR, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(x_labels_short)
ax.set_ylabel("Decode Throughput (tok/s)"); ax.set_title("Throughput"); ax.legend(fontsize=9)

# Bottom-right: Max context (big numbers)
ax = axes[1, 1]
ax.axis("off")
ctx_multiplier = max_ctx_tq / max_ctx_baseline if max_ctx_baseline > 0 else float("nan")
ax.text(0.5, 0.75, "Max Context Before OOM", ha="center", va="center",
        fontsize=13, fontweight="bold", transform=ax.transAxes)
ax.text(0.5, 0.55, f"Baseline:  {max_ctx_baseline:,} tokens", ha="center", va="center",
        fontsize=11, color=BL_COLOR, transform=ax.transAxes)
ax.text(0.5, 0.40, f"TurboQuant: {max_ctx_tq:,} tokens", ha="center", va="center",
        fontsize=11, color=TQ_COLOR, transform=ax.transAxes)
ax.text(0.5, 0.22, f"{ctx_multiplier:.2f}× more context", ha="center", va="center",
        fontsize=18, fontweight="bold", color="#333", transform=ax.transAxes)

plt.tight_layout()
plt.savefig(f"{CONFIG['results_dir']}/plot_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

## Section 7: Conclusions

The cells below extract the key numbers and auto-fill a summary table.

In [ ]:
from IPython.display import Markdown, display

# Extract values at 32K (or the largest available context)
largest_ctx = max(c for c in ctx_lengths if c in bl and c in tq)

bl_vram_32k = bl[largest_ctx]["peak_vram_mb"]
tq_vram_32k = tq[largest_ctx]["peak_vram_mb"]
vram_saved_mb = bl_vram_32k - tq_vram_32k
vram_saved_pct = 100 * vram_saved_mb / bl_vram_32k if bl_vram_32k else 0

bl_ppl_32k = bl[largest_ctx]["perplexity"]
tq_ppl_32k = tq[largest_ctx]["perplexity"]
ppl_delta = tq_ppl_32k - bl_ppl_32k

bl_toks_32k = bl[largest_ctx]["decode_tps"]
tq_toks_32k = tq[largest_ctx]["decode_tps"]
speed_delta_pct = 100 * (tq_toks_32k - bl_toks_32k) / bl_toks_32k if bl_toks_32k else 0

ctx_label = f"{largest_ctx // 1024}K"
ctx_mult_str = f"{ctx_multiplier:.2f}" if ctx_multiplier == ctx_multiplier else "N/A"  # NaN check

summary_md = f"""
## Results Summary

| Metric | Baseline | TurboQuant | Delta |
|--------|----------|------------|-------|
| Peak VRAM @ {ctx_label} | {bl_vram_32k:,} MB | {tq_vram_32k:,} MB | {'-' if vram_saved_mb >= 0 else '+'}{abs(vram_saved_mb):,.0f} MB ({vram_saved_pct:.0f}%) |
| Perplexity @ {ctx_label} | {bl_ppl_32k:.3f} | {tq_ppl_32k:.3f} | {'+' if ppl_delta >= 0 else ''}{ppl_delta:.3f} |
| Decode tok/s @ {ctx_label} | {bl_toks_32k:.1f} | {tq_toks_32k:.1f} | {'+' if speed_delta_pct >= 0 else ''}{speed_delta_pct:.0f}% |
| Max context before OOM | {max_ctx_baseline:,} tokens | {max_ctx_tq:,} tokens | {ctx_mult_str}× |

**Bottom line**: TurboQuant achieves **{vram_saved_pct:.0f}% VRAM reduction** at {ctx_label} context
with only **+{ppl_delta:.3f} perplexity increase** ({100 * ppl_delta / bl_ppl_32k:.1f}% relative),
enabling **{ctx_mult_str}× longer context** on the same RTX 3090.

TQ key bits: {CONFIG['tq_key_bits']} | TQ value bits: {CONFIG['tq_value_bits']} | 
Model: `{CONFIG['model_id']}`
"""

display(Markdown(summary_md))

## Section 8: Save Results & Export

All raw data and plots are saved to the `results/` directory.

In [ ]:
import json
import os
from datetime import datetime

os.makedirs(CONFIG["results_dir"], exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Build serializable results dict (strip raw_reps for compactness)
def strip_reps(results_list):
    out = []
    for r in results_list:
        row = {k: v for k, v in r.items() if k != "raw_reps"}
        out.append(row)
    return out

all_results = {
    "timestamp": timestamp,
    "config": CONFIG,
    "baseline": strip_reps(baseline_results),
    "turboquant": strip_reps(tq_results),
    "oom": {
        "max_ctx_baseline": max_ctx_baseline,
        "max_ctx_tq": max_ctx_tq,
        "multiplier": ctx_multiplier,
    },
}

json_path = f"{CONFIG['results_dir']}/benchmark_{timestamp}.json"
with open(json_path, "w") as f:
    json.dump(all_results, f, indent=2)

print(f"Results saved to: {json_path}")
print()
print("Plots saved:")
for fname in [
    "plot_vram_vs_context.png",
    "plot_perplexity.png",
    "plot_latency.png",
    "plot_dashboard.png",
]:
    full_path = f"{CONFIG['results_dir']}/{fname}"
    exists = os.path.exists(full_path)
    print(f"  {'OK' if exists else 'MISSING'}  {full_path}")

print()
print("All results saved.")